# ccdp — YOLOv8-seg quick wins on Colab

Two fast, deployable-stack training runs that target the same cost-accuracy goal as Mask R-CNN, without the heavy
two-stage model:

1. **CarDD damage masks** — `ccdp train detector --seg` trains YOLOv8-seg on CarDD's polygons → a true
   damaged-area % (boxes over-estimate area).
2. **Car-parts segmenter** — `ccdp train parts` trains YOLOv8-seg on Ultralytics **carparts-seg** (auto-downloads)
   → a learned part map that replaces the heuristic `infer_part_from_damage` (the cost catalog is parts-keyed).

Both are single-stage YOLO, so they're cheap on a T4 and drop into the detector pipeline already deployed.

> **Runtime → Change runtime type → T4 GPU** first.


## 1. GPU check


In [ ]:
import torch
print('CUDA:',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '— set T4')

## 2. Clone + install


In [ ]:
%cd /content
![ -d car-crash-fix-amount-predictor] || git clone https://github.com/theDocWho/car-crash-fix-amount-predictor.git
%cd /content/car-crash-fix-amount-predictor
!git checkout checkpoint-18-yolov8-seg-quickwins 2>/dev/null || echo 'using current branch'
!pip -q install -e '.[ml]'

## 3. SSL fix (for any pretrained-weight downloads)


In [ ]:
import os, certifi
os.environ['SSL_CERT_FILE']=certifi.where()
os.environ['REQUESTS_CA_BUNDLE']=certifi.where()
print('ok')

## 4. Mount Drive + persist checkpoints/ and data/


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, pathlib
BASE='/content/drive/MyDrive/ccdp'
for sub in ('checkpoints','data'):
    os.makedirs(f'{BASE}/{sub}',exist_ok=True); p=pathlib.Path(sub)
    if not p.is_symlink():
        if p.exists(): os.system(f'cp -rn {sub}/* {BASE}/{sub}/ 2>/dev/null; rm -rf {sub}')
        os.symlink(f'{BASE}/{sub}',sub)
print('checkpoints ->',os.path.realpath('checkpoints'))

## 5. Kaggle auth + CarDD (only QW1 needs it; QW2 auto-downloads)


In [ ]:
from google.colab import files; files.upload()  # kaggle.json
!mkdir -p ~/.kaggle && mv -f kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!bash scripts/download_datasets.sh
!ccdp costing init || true

## Quick win 1 — CarDD YOLOv8-seg (damage masks)

Builds the polygon dataset from CarDD's COCO masks and trains the seg model. ~1–2 h on a T4 for 60 epochs.


In [ ]:
!ccdp train detector --seg --epochs 60 --batch 16 --workers 2

## Quick win 2 — car-parts segmenter (carparts-seg, auto-downloads)

`carparts-seg` (23 part classes) is fetched automatically by Ultralytics — no Kaggle needed. ~1–2 h on a T4.


In [ ]:
!ccdp train parts --epochs 60 --batch 16 --workers 2

## (Optional) Quick win 3 — bigger detector (YOLOv8s boxes)

Same pipeline, a larger model for a few extra points of mAP.


In [ ]:
!ccdp train detector --model yolov8s.pt --epochs 60 --batch 16 --workers 2 --tag yolov8s

## Promote + inspect


In [ ]:
!ccdp registry list --variant yoloseg
!ccdp registry list --variant parts
# promote (replace <run_id> with ids above):
# !ccdp registry promote <run_id> yoloseg
# !ccdp registry promote <run_id> parts

## Package weights to Drive


In [ ]:
import os, shutil
OUT='/content/drive/MyDrive/ccdp/release_yoloseg'; os.makedirs(OUT,exist_ok=True)
def cp(src,dst):
    if os.path.exists(src): shutil.copy(os.path.realpath(src),f'{OUT}/{dst}'); print('packaged',dst)
    else: print('MISSING (promote first):',src)
cp('checkpoints/production/yoloseg.pt','yoloseg.pt')
cp('checkpoints/production/parts.pt','parts.pt')
print('\nDownload from Drive:',OUT)

## Next step (separate PR)

These produce the weights; wiring them into inference — using YOLOv8-seg mask area % as an XGBoost feature, and
overlapping damage masks with the parts map for parts-keyed costing — is a follow-up integration on top of the
deployed stack. Download the weights above and attach them to a GitHub release when that wiring lands.
